# DMET-SSTKG — Motif-Extractor Fixed-Budget Sensitivity

This notebook reproduces the source-validation architecture sensitivity analysis reported in **Table 5** of the manuscript.

Seven matched configurations are trained under the same five-epoch budget. The analysis isolates phase-checkpoint initialization, first-layer trainability, temporal window length, and stride. The saved source split is reused unchanged and remote outcomes are not used for selection.

Outputs are written under `outputs/validation_analyses/motif_extractor_fixed_budget_sensitivity/`.


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ROLE-NORMALIZED S-STKG MOTIF EXTRACTOR / MOTIF HEAD TRAINING
============================================================

Use after the clean role-normalized phase classifier.

Key differences from the old motif script:
- Uses SurgicalSTKG_20CASE_ROLE.
- Uses the SAME data_split.npz from the clean 20-case 14/3/3 phase run.
- Loads the clean role-normalized phase checkpoint.
- Removes raw case_id from node input features by setting column 7 to 0.0.
- Keeps g_ratio and p_ratio by default.
- Saves best_motif_model.pt and data_split.npz for later hub construction.
"""

import os
from pathlib import Path
import json
import shutil
import random
import gc
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GATv2Conv
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    silhouette_score,
)
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE


# =========================================================
# 0) CONFIG — REPOSITORY-RELATIVE PATHS
# =========================================================
SEED = 42

def find_repo_root(start=None):
    """Locate the DMET-SSTKG repository root from the current working directory."""
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "Codes").exists():
            return candidate
    raise FileNotFoundError(
        "DMET-SSTKG repository root not found. Start Jupyter from the repository root "
        "or from a subdirectory inside the repository."
    )


def first_existing_file(candidates, glob_patterns=()):
    """Resolve a generated artifact without embedding machine-specific absolute paths."""
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.is_file():
            return candidate
    for pattern in glob_patterns:
        matches = sorted(REPO_ROOT.glob(pattern))
        if matches:
            return matches[-1]
    # Return the first documented candidate so the preflight error is informative.
    return Path(candidates[0])

REPO_ROOT = find_repo_root()
ARTIFACT_DIR = REPO_ROOT / "artifacts"

BASE_PATH = str(
    REPO_ROOT / "S-STKG_Data" / "SurgicalSTKG_Local_Cholec80_M2CAI"
)
BIN_PATH = str(REPO_ROOT / "Binary_Cache_HUB_Cholec80_M2cai")

PHASE_CKPT = first_existing_file(
    [
        ARTIFACT_DIR / "best_sstkg_phase_classifier.pt",
        ARTIFACT_DIR / "best_robust_model.pt",
    ],
    [
        "outputs/phase_classifier_runs/*/best_sstkg_phase_classifier.pt",
        "outputs/phase_classifier_runs/*/best_robust_model.pt",
    ],
)
PHASE_RESULTS_DIR = str(PHASE_CKPT.parent)
SPLIT_NPZ = str(PHASE_CKPT.parent / "data_split.npz")

FORCE_REPROCESS_CACHE = False

CHUNK_SIZE = 100
NUM_PHASES = 8
HIDDEN_DIM = 128
MOTIF_DIM = 128
NUM_EPOCHS = 5

SCREEN_TRAIN_FRACTION = 1.0
SCREEN_SEED = 42

AUX_WEIGHT = 0.02
SEQ_WEIGHT = 0.20
LABEL_SMOOTHING = 0.05

MOTIF_LR = 5e-4
BACKBONE_LR = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

REMOVE_CASE_ID_FEATURE = True
REMOVE_G_RATIO_FEATURE = False
REMOVE_P_RATIO_FEATURE = False
CASE_ID_COL = 7
G_RATIO_COL = 8
P_RATIO_COL = 9

PHASE_NAMES = [
    "TrocarPlacement",
    "Preparation",
    "CalotTriangleDissection",
    "ClippingCutting",
    "GallbladderDissection",
    "GallbladderPackaging",
    "CleaningCoagulation",
    "GallbladderRetraction",
]

# =========================================================
# 1) HELPERS
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def safe_label_to_zero_based(raw_label, num_classes=8):
    raw_label = int(raw_label)
    if 1 <= raw_label <= num_classes:
        return raw_label - 1
    if 0 <= raw_label < num_classes:
        return raw_label
    raise ValueError(f"Unexpected phase label value: {raw_label}")


# =========================================================
# 2) DATASET
# =========================================================
class SurgicalTurboDataset(Dataset):
    def __init__(self, root_dir, binary_dir, chunk_size=100, force_reprocess=False):
        super().__init__(root_dir)
        self.root_dir = root_dir
        self.binary_dir = binary_dir
        self.chunk_size = chunk_size

        if force_reprocess:
            print(f"[INFO] Removing cache: {self.binary_dir}")
            if os.path.exists(self.binary_dir):
                shutil.rmtree(self.binary_dir)
            os.makedirs(self.binary_dir, exist_ok=True)
            self._convert_and_fragment(root_dir)
        elif not os.path.exists(self.binary_dir) or len(os.listdir(self.binary_dir)) == 0:
            print(f"[INFO] Cache missing. Building: {self.binary_dir}")
            os.makedirs(self.binary_dir, exist_ok=True)
            self._convert_and_fragment(root_dir)
        else:
            print(f"[OK] Existing cache is used: {self.binary_dir}")

        self.files = sorted([f for f in os.listdir(self.binary_dir) if f.endswith(".pt")])
        if len(self.files) == 0:
            raise RuntimeError(f"Binary cache is empty: {self.binary_dir}")

    def _clean_node_features(self, f_n):
        f_n = np.asarray(f_n).copy()
        if f_n.ndim != 2 or f_n.shape[1] < 11:
            raise ValueError(f"Unexpected node feature shape: {f_n.shape}. Expected at least 11 columns.")
        if REMOVE_CASE_ID_FEATURE:
            f_n[:, CASE_ID_COL] = 0.0
        if REMOVE_G_RATIO_FEATURE:
            f_n[:, G_RATIO_COL] = 0.0
        if REMOVE_P_RATIO_FEATURE:
            f_n[:, P_RATIO_COL] = 0.0
        return f_n

    def _convert_and_fragment(self, root_dir):
        print("[INFO] Converting ROLE-normalized STKG into binary fragments...")
        if not os.path.exists(root_dir):
            raise FileNotFoundError(f"BASE_PATH not found: {root_dir}")
        cases = sorted([d for d in os.listdir(root_dir) if d.startswith("Case_")])
        if not cases:
            raise RuntimeError(f"No Case_ folders found under BASE_PATH: {root_dir}")

        global_idx = 0
        for case in tqdm(cases, desc="Building role motif cache"):
            case_path = os.path.join(root_dir, case)
            prefixes = set([f.replace("_nodes.txt", "") for f in os.listdir(case_path) if f.endswith("_nodes.txt")])

            for pref in sorted(prefixes):
                nodes_path = os.path.join(case_path, f"{pref}_nodes.txt")
                edges_path = os.path.join(case_path, f"{pref}_edges.txt")
                edge_feat_path = os.path.join(case_path, f"{pref}_edge_features.txt")
                label_path = os.path.join(case_path, f"{pref}_labels.txt")
                required = [nodes_path, edges_path, edge_feat_path, label_path]
                if not all(os.path.exists(p) for p in required):
                    print(f"[WARN] Missing files for prefix {pref}, skipping.")
                    continue

                n_df = pd.read_csv(nodes_path)
                e_df = pd.read_csv(edges_path)
                ef_df = pd.read_csv(edge_feat_path)
                raw_label = int(pd.read_csv(label_path).iloc[0, 1])
                label = safe_label_to_zero_based(raw_label, NUM_PHASES)
                unique_grs = sorted(n_df.iloc[:, G_RATIO_COL].unique())

                for i in range(0, len(unique_grs), self.chunk_size):
                    chunk_grs = unique_grs[i:i + self.chunk_size]
                    if len(chunk_grs) < 10:
                        continue

                    graph_seq = []
                    for gr in chunk_grs:
                        f_n_raw = n_df[n_df.iloc[:, G_RATIO_COL] == gr].values
                        f_n = self._clean_node_features(f_n_raw)
                        f_e = e_df[e_df.iloc[:, 0] == gr].values
                        f_ef = ef_df[ef_df.iloc[:, 0] == gr].values

                        e_idx = (
                            torch.tensor(f_e[:, 2:4].astype(np.int64), dtype=torch.long).t().contiguous()
                            if len(f_e) > 0 else torch.zeros((2, 0), dtype=torch.long)
                        )
                        e_at = (
                            torch.tensor(f_ef[:, 4:7], dtype=torch.float)
                            if len(f_ef) > 0 else torch.zeros((0, 3), dtype=torch.float)
                        )
                        data = Data(
                            x=torch.tensor(f_n, dtype=torch.float),
                            edge_index=e_idx,
                            edge_attr=e_at,
                            y=torch.tensor([label], dtype=torch.long),
                        )
                        data.is_v = data.x[:, 10] == 1
                        graph_seq.append(data)

                    torch.save(
                        {
                            "graph_seq": graph_seq,
                            "metadata": {
                                "case_id": str(case),
                                "prefix": str(pref),
                                "chunk_start_gr": int(chunk_grs[0]),
                                "chunk_end_gr": int(chunk_grs[-1]),
                                "phase_label_id": int(label),
                                "raw_phase_label": int(raw_label),
                                "feature_cleaning": {
                                    "case_id_zeroed": bool(REMOVE_CASE_ID_FEATURE),
                                    "g_ratio_zeroed": bool(REMOVE_G_RATIO_FEATURE),
                                    "p_ratio_zeroed": bool(REMOVE_P_RATIO_FEATURE),
                                },
                            },
                        },
                        os.path.join(self.binary_dir, f"frag_{global_idx:06d}.pt")
                    )
                    global_idx += 1
        print(f"[OK] Binary fragments created: {global_idx}")

    def len(self):
        return len(self.files)

    def _load_raw(self, idx):
        return torch.load(os.path.join(self.binary_dir, self.files[int(idx)]), weights_only=False)

    def get_with_meta(self, idx):
        obj = self._load_raw(idx)
        if isinstance(obj, dict) and "graph_seq" in obj:
            meta = dict(obj.get("metadata", {}) or {})
            meta["fragment_file"] = self.files[int(idx)]
            return obj["graph_seq"], meta
        return obj, {"fragment_file": self.files[int(idx)], "case_id": "UNKNOWN"}

    def get(self, idx):
        seq, _ = self.get_with_meta(idx)
        return seq


# =========================================================
# 3) SPLIT
# =========================================================
def load_saved_split(split_npz, dataset_len):
    if not os.path.exists(split_npz):
        raise FileNotFoundError(f"SPLIT_NPZ not found: {split_npz}")
    s = np.load(split_npz)
    train_idx = s["train_idx"].astype(int)
    val_idx = s["val_idx"].astype(int)
    test_idx = s["test_idx"].astype(int)
    all_idx = np.concatenate([train_idx, val_idx, test_idx])
    if all_idx.min() < 0 or all_idx.max() >= dataset_len:
        raise ValueError(f"Split incompatible with dataset length={dataset_len}. Min={all_idx.min()}, Max={all_idx.max()}")
    if set(train_idx) & set(val_idx) or set(train_idx) & set(test_idx) or set(val_idx) & set(test_idx):
        raise ValueError("Split overlap detected.")
    return train_idx, val_idx, test_idx


def summarize_split(dataset, train_idx, val_idx, test_idx, save_dir):
    rows = []
    for split_name, indices in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
        for idx in indices:
            seq, meta = dataset.get_with_meta(int(idx))
            y = int(seq[0].y.item())
            rows.append({
                "split": split_name,
                "idx": int(idx),
                "case_id": str(meta.get("case_id", "UNKNOWN")),
                "phase_id": y,
                "phase_name": PHASE_NAMES[y],
                "fragment_file": str(meta.get("fragment_file", "")),
                "prefix": str(meta.get("prefix", "")),
            })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(save_dir, "motif_split_fragment_metadata.csv"), index=False)
    case_summary = df.groupby(["split", "case_id"], as_index=False).agg(num_fragments=("idx", "count"))
    case_summary.to_csv(os.path.join(save_dir, "motif_split_case_summary.csv"), index=False)
    phase_counts = df.groupby(["split", "phase_name"], as_index=False).agg(num_fragments=("idx", "count"))
    phase_counts.to_csv(os.path.join(save_dir, "motif_split_phase_counts.csv"), index=False)
    print("\n[SPLIT CASE SUMMARY]")
    print(case_summary)
    print("\n[SPLIT PHASE COUNTS]")
    print(phase_counts.pivot(index="phase_name", columns="split", values="num_fragments").fillna(0).astype(int))


# =========================================================
# 4) CLASS WEIGHTS
# =========================================================
def compute_motif_class_weights(dataset, indices, motif_window=16, motif_stride=8, num_classes=8):
    counts = np.zeros(num_classes, dtype=np.float64)
    for idx in indices:
        seq = dataset.get(int(idx))
        label = int(seq[0].y.item())
        T = len(seq)
        n_motifs = 1 if T < motif_window else len(range(0, T - motif_window + 1, motif_stride))
        counts[label] += n_motifs
    counts = np.clip(counts, a_min=1, a_max=None)
    weights = 1.0 / counts
    weights = weights / weights.sum() * num_classes
    print("\n[MOTIF WINDOW COUNTS]", counts.astype(int).tolist())
    print("[MOTIF CLASS WEIGHTS]", [round(float(w), 4) for w in weights.tolist()])
    return torch.tensor(weights, dtype=torch.float)


# =========================================================
# 5) MODEL
# =========================================================
class RobustMosaicGNN(nn.Module):
    def __init__(self, in_channels=11, edge_dim=3, hidden_dim=128, num_phases=8, motif_dim=128):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.conv1 = GATv2Conv(in_channels, hidden_dim, edge_dim=edge_dim)
        self.conv2 = GATv2Conv(hidden_dim, hidden_dim, edge_dim=edge_dim)
        self.context_attn = nn.Linear(hidden_dim * 2, 1)
        self.rnn = nn.GRU(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 3, num_phases)
        self.motif_proj = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, motif_dim))
        self.motif_cls = nn.Linear(motif_dim, num_phases)
        self.window_fuse = nn.Linear(hidden_dim * 2, hidden_dim)
        self.motif_window = 16
        self.motif_stride = 8

    def encode_frames(self, graph_seq):
        frame_embs = []
        for data in graph_seq:
            x = self.conv1(data.x, data.edge_index, data.edge_attr).relu()
            x = self.conv2(x, data.edge_index, data.edge_attr).relu()
            mean_emb = x.mean(dim=0, keepdim=True)
            v_node_emb = x[data.is_v].mean(dim=0, keepdim=True) if data.is_v.any() else mean_emb
            attn_weight = torch.sigmoid(self.context_attn(torch.cat([mean_emb, v_node_emb], dim=-1)))
            frame_embs.append(attn_weight * v_node_emb + (1.0 - attn_weight) * mean_emb)
        return torch.stack(frame_embs).squeeze(1)

    def apply_temporal_modeling(self, frame_tensor):
        rnn_out, h_n = self.rnn(frame_tensor.unsqueeze(0))
        rnn_out = rnn_out.squeeze(0)
        temporal_out = 0.5 * (rnn_out[:, :self.hidden_dim] + rnn_out[:, self.hidden_dim:])
        temporal_frames = 0.5 * frame_tensor + 0.5 * temporal_out
        rnn_final = torch.cat([h_n[0], h_n[1]], dim=-1)
        global_skip = frame_tensor.mean(dim=0, keepdim=True)
        seq_feature = torch.cat([rnn_final, global_skip], dim=-1)
        return temporal_frames, seq_feature

    def pool_window(self, window_tensor):
        return self.window_fuse(torch.cat([window_tensor.mean(dim=0, keepdim=True), window_tensor[-1].unsqueeze(0)], dim=-1))

    def extract_motifs(self, frame_tensor):
        T = frame_tensor.shape[0]
        if T < self.motif_window:
            return F.normalize(self.motif_proj(self.pool_window(frame_tensor)), dim=-1)
        windows = [self.pool_window(frame_tensor[s:s + self.motif_window]) for s in range(0, T - self.motif_window + 1, self.motif_stride)]
        return F.normalize(self.motif_proj(torch.cat(windows, dim=0)), dim=-1)

    def forward(self, graph_seq):
        frames = self.encode_frames(graph_seq)
        temporal_frames, seq_feature = self.apply_temporal_modeling(frames)
        sigs = self.extract_motifs(temporal_frames)
        return {"motif_logits": self.motif_cls(sigs), "motif_sigs": sigs, "seq_logits": self.fc(seq_feature)}


# =========================================================
# 6) LOSSES / OPTIMIZER
# =========================================================
def motif_auxiliary_loss(sigs, threshold=0.7):
    if sigs.size(0) <= 1:
        return torch.tensor(0.0, device=sigs.device)
    pull = (1.0 - F.cosine_similarity(sigs[:-1], sigs[1:], dim=-1)).mean()
    sim_mat = sigs @ sigs.t()
    mask = ~torch.eye(sim_mat.size(0), device=sigs.device, dtype=torch.bool)
    off_diag = sim_mat[mask]
    push = torch.relu(off_diag - threshold).mean() if off_diag.numel() > 0 else torch.tensor(0.0, device=sigs.device)
    return pull + 0.5 * push


def compute_total_motif_loss(out, motif_target, seq_target, criterion, aux_weight=0.02, seq_weight=0.20):
    motif_cls_loss = criterion(out["motif_logits"], motif_target)
    motif_aux_loss = motif_auxiliary_loss(out["motif_sigs"])
    seq_cls_loss = criterion(out["seq_logits"], seq_target)
    total_loss = motif_cls_loss + aux_weight * motif_aux_loss + seq_weight * seq_cls_loss
    return total_loss, motif_cls_loss, motif_aux_loss, seq_cls_loss


def apply_partial_unfreeze(model):
    for p in model.parameters():
        p.requires_grad = False
    for name, p in model.named_parameters():
        if ("motif_" in name or "conv2" in name or "context_attn" in name or "rnn" in name or "window_fuse" in name or "fc" in name):
            p.requires_grad = True
    print("\n[TRAINABLE PARAMS]")
    for name, p in model.named_parameters():
        if p.requires_grad:
            print("  -", name)


def build_optimizer(model):
    motif_params, backbone_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "motif_" in name or "window_fuse" in name or "fc" in name:
            motif_params.append(p)
        else:
            backbone_params.append(p)
    return torch.optim.AdamW([
        {"params": motif_params, "lr": MOTIF_LR},
        {"params": backbone_params, "lr": BACKBONE_LR},
    ], weight_decay=WEIGHT_DECAY)


# =========================================================
# 7) METRICS
# =========================================================
def compute_basic_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    return {"accuracy": acc, "macro_precision": macro_p, "macro_recall": macro_r, "macro_f1": macro_f1, "weighted_precision": weighted_p, "weighted_recall": weighted_r, "weighted_f1": weighted_f1}


def evaluate_split(model, dataset, indices, device, criterion, phase_names, aux_weight=0.02, seq_weight=0.20):
    model.eval()
    losses, motif_losses, aux_losses, seq_losses = [], [], [], []
    y_true, y_pred = [], []
    with torch.no_grad():
        for idx in indices:
            seq = [d.to(device) for d in dataset.get(int(idx))]
            out = model(seq)
            motif_target = seq[0].y.to(device).repeat(out["motif_logits"].size(0))
            seq_target = seq[0].y.to(device)
            total, motif_l, aux_l, seq_l = compute_total_motif_loss(out, motif_target, seq_target, criterion, aux_weight, seq_weight)
            losses.append(total.item()); motif_losses.append(motif_l.item()); aux_losses.append(aux_l.item()); seq_losses.append(seq_l.item())
            preds = out["motif_logits"].argmax(dim=1).cpu().numpy()
            y_pred.extend(preds)
            y_true.extend([seq[0].y.item()] * len(preds))
    y_true = np.array(y_true); y_pred = np.array(y_pred)
    metrics = compute_basic_metrics(y_true, y_pred)
    metrics.update({
        "loss_total": float(np.mean(losses)) if losses else 0.0,
        "loss_motif_cls": float(np.mean(motif_losses)) if motif_losses else 0.0,
        "loss_aux": float(np.mean(aux_losses)) if aux_losses else 0.0,
        "loss_seq_cls": float(np.mean(seq_losses)) if seq_losses else 0.0,
        "y_true": y_true,
        "y_pred": y_pred,
    })
    metrics["report"] = classification_report(y_true, y_pred, labels=list(range(len(phase_names))), target_names=phase_names, zero_division=0)
    return metrics


def safe_silhouette(x, labels, metric="cosine", max_samples=4000, random_state=42):
    labels = np.asarray(labels)
    if x is None or len(x) < 2 or len(np.unique(labels)) < 2:
        return np.nan
    sample_size = max_samples if len(x) > max_samples else None
    return float(silhouette_score(x, labels, metric=metric, sample_size=sample_size, random_state=random_state))


def analyze_motif_representation(model, dataset, indices, device):
    model.eval()
    all_sigs, all_labels = [], []
    with torch.no_grad():
        for idx in indices:
            seq = [d.to(device) for d in dataset.get(int(idx))]
            out = model(seq)
            all_sigs.append(out["motif_sigs"].cpu().numpy())
            all_labels.extend([seq[0].y.item()] * out["motif_sigs"].size(0))
    all_sigs = np.concatenate(all_sigs, axis=0)
    all_labels = np.array(all_labels)
    return safe_silhouette(all_sigs, all_labels, metric="cosine"), all_sigs, all_labels


def analyze_unsupervised_structure_from_sigs(sigs, n_clusters=8, random_state=42):
    if sigs.shape[0] < n_clusters or n_clusters < 2:
        return np.nan, None, None
    kmeans = KMeans(n_clusters=n_clusters, n_init=20, random_state=random_state)
    cluster_ids = kmeans.fit_predict(sigs)
    return safe_silhouette(sigs, cluster_ids, metric="cosine"), cluster_ids, np.bincount(cluster_ids, minlength=n_clusters)


def plot_confusion_matrix(cm, phase_names, save_path, title):
    plt.figure(figsize=(11, 9))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=phase_names, yticklabels=phase_names)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()


def plot_history(history, save_path):
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(epochs, history["train_loss"], marker="o", label="Train Total Loss")
    axes[0].plot(epochs, history["val_loss"], marker="s", label="Val Total Loss")
    axes[0].set_title("Total Loss"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(epochs, history["train_acc"], marker="o", label="Train Acc")
    axes[1].plot(epochs, history["val_acc"], marker="s", label="Val Acc")
    axes[1].set_title("Accuracy (%)"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=300); plt.close()



# =========================================================
# 8) MOTIF-EXTRACTOR FIXED-BUDGET SENSITIVITY
# =========================================================

# Existing retained motif checkpoint: used only as the baseline comparator.
# No original checkpoint is overwritten by this audit.
REPORTED_MOTIF_CKPT = first_existing_file(
    [
        ARTIFACT_DIR / "best_phase_initialized_motif_extractor.pt",
        ARTIFACT_DIR / "best_motif_model.pt",
    ],
    [
        "outputs/motif_extractor_runs/*/best_phase_initialized_motif_extractor.pt",
        "outputs/motif_extractor_runs/*/best_motif_model.pt",
    ],
)
REPORTED_MOTIF_RESULTS_DIR = str(REPORTED_MOTIF_CKPT.parent)
AUDIT_RESULTS_DIR = str(
    REPO_ROOT / "outputs" / "analysis_audits" / "motif_extractor_fixed_budget_sensitivity"
)

# Seven matched fixed-budget configurations. No factorial combination search.
AUDIT_CONFIGS = [
    {
        "config_id": "baseline_L16_S8_phaseinit_frozen",
        "comparison_group": "baseline",
        "initialization": "phase_checkpoint",
        "freeze_first_gatv2": True,
        "motif_window": 16,
        "motif_stride": 8,
    },
    {
        "config_id": "phase_init_full_unfreeze",
        "comparison_group": "freeze_and_initialization",
        "initialization": "phase_checkpoint",
        "freeze_first_gatv2": False,
        "motif_window": 16,
        "motif_stride": 8,
    },
    {
        "config_id": "random_init_full_unfreeze",
        "comparison_group": "initialization",
        "initialization": "random",
        "freeze_first_gatv2": False,
        "motif_window": 16,
        "motif_stride": 8,
    },
    {
        "config_id": "window_8_stride_8",
        "comparison_group": "window_length",
        "initialization": "phase_checkpoint",
        "freeze_first_gatv2": True,
        "motif_window": 8,
        "motif_stride": 8,
    },
    {
        "config_id": "window_24_stride_8",
        "comparison_group": "window_length",
        "initialization": "phase_checkpoint",
        "freeze_first_gatv2": True,
        "motif_window": 24,
        "motif_stride": 8,
    },
    {
        "config_id": "window_16_stride_4",
        "comparison_group": "stride",
        "initialization": "phase_checkpoint",
        "freeze_first_gatv2": True,
        "motif_window": 16,
        "motif_stride": 4,
    },
    {
        "config_id": "window_16_stride_16",
        "comparison_group": "stride",
        "initialization": "phase_checkpoint",
        "freeze_first_gatv2": True,
        "motif_window": 16,
        "motif_stride": 16,
    },
]



def configure_trainability_for_audit(model, freeze_first_gatv2):
    # Retained policy: conv1 frozen, upper graph-temporal + motif modules trainable.
    if freeze_first_gatv2:
        apply_partial_unfreeze(model)
        return
    # Full unfreeze differs from the retained policy only by allowing conv1 to update.
    for p in model.parameters():
        p.requires_grad = True


def build_audit_model(cfg):
    set_seed(SEED)
    model = RobustMosaicGNN(
        hidden_dim=HIDDEN_DIM,
        num_phases=NUM_PHASES,
        motif_dim=MOTIF_DIM,
    ).to(DEVICE)
    model.motif_window = int(cfg["motif_window"])
    model.motif_stride = int(cfg["motif_stride"])

    if cfg["initialization"] == "phase_checkpoint":
        ckpt = torch.load(PHASE_CKPT, map_location=DEVICE, weights_only=False)
        msg = model.load_state_dict(ckpt["model_state_dict"], strict=False)
        print(f"[{cfg['config_id']}] phase checkpoint loaded | missing={len(msg.missing_keys)} unexpected={len(msg.unexpected_keys)}")
    elif cfg["initialization"] == "random":
        print(f"[{cfg['config_id']}] random initialization")
    else:
        raise ValueError(cfg["initialization"])

    configure_trainability_for_audit(model, bool(cfg["freeze_first_gatv2"]))
    return model


def train_one_audit_variant(cfg, dataset, train_idx, val_idx, out_dir):
    print("\n" + "=" * 100)
    print("RUNNING", cfg)
    print("=" * 100)

    config_id = cfg["config_id"]
    resume_path = os.path.join(
        out_dir,
        f"{config_id}_RESUME.pt",
    )
    history_path = os.path.join(
        out_dir,
        f"{config_id}_history.csv",
    )

    # Reset before construction so all phase-initialized variants receive the
    # same newly initialized motif-specific layers.
    set_seed(SCREEN_SEED)
    model = build_audit_model(cfg)

    weights = compute_motif_class_weights(
        dataset,
        train_idx,
        motif_window=model.motif_window,
        motif_stride=model.motif_stride,
        num_classes=NUM_PHASES,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(
        weight=weights,
        label_smoothing=LABEL_SMOOTHING,
    )
    optimizer = build_optimizer(model)

    best_val_macro_f1 = -1.0
    best_val_acc = -1.0
    best_epoch = -1
    best_state = None
    history = []
    start_epoch = 1

    if os.path.exists(resume_path):
        resume = torch.load(
            resume_path,
            map_location=DEVICE,
            weights_only=False,
        )
        if resume.get("config", {}) != cfg:
            raise RuntimeError(
                f"Resume checkpoint configuration mismatch for {config_id}"
            )

        model.load_state_dict(
            resume["model_state_dict"],
            strict=True,
        )
        optimizer.load_state_dict(
            resume["optimizer_state_dict"]
        )
        best_val_macro_f1 = float(
            resume["best_val_macro_f1"]
        )
        best_val_acc = float(
            resume["best_val_acc"]
        )
        best_epoch = int(
            resume["best_epoch"]
        )
        best_state = resume.get(
            "best_state_dict",
            None,
        )
        history = list(
            resume.get("history", [])
        )
        start_epoch = int(
            resume["completed_epoch"]
        ) + 1

        print(
            f"[RESUME] {config_id}: continuing from epoch {start_epoch}"
        )

    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        model.train()

        # Same deterministic fragment order for all configurations and epochs.
        pbar = tqdm(
            train_idx,
            desc=f"{config_id} | Epoch {epoch:02d}",
        )

        running_loss = []

        for idx in pbar:
            seq = [
                d.to(DEVICE)
                for d in dataset.get(int(idx))
            ]

            optimizer.zero_grad(
                set_to_none=True
            )

            out = model(seq)

            motif_target = (
                seq[0].y.to(DEVICE)
                .repeat(
                    out["motif_logits"].size(0)
                )
            )
            seq_target = seq[0].y.to(DEVICE)

            total_loss, _, _, _ = compute_total_motif_loss(
                out,
                motif_target,
                seq_target,
                criterion,
                AUX_WEIGHT,
                SEQ_WEIGHT,
            )

            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=GRAD_CLIP_NORM,
            )

            optimizer.step()

            running_loss.append(
                float(total_loss.item())
            )

            del seq, out, motif_target, seq_target, total_loss

        val_metrics = evaluate_split(
            model,
            dataset,
            val_idx,
            DEVICE,
            criterion,
            PHASE_NAMES,
            AUX_WEIGHT,
            SEQ_WEIGHT,
        )

        val_macro = float(
            val_metrics["macro_f1"]
        )
        val_acc = float(
            val_metrics["accuracy"] * 100.0
        )
        mean_train_loss = float(
            np.mean(running_loss)
        )

        history.append(
            {
                "config_id": config_id,
                "epoch": int(epoch),
                "mean_train_loss": mean_train_loss,
                "val_macro_f1": val_macro,
                "val_accuracy_pct": val_acc,
            }
        )

        pd.DataFrame(history).to_csv(
            history_path,
            index=False,
        )

        print(
            f"[{config_id}] "
            f"epoch={epoch:02d} "
            f"train_loss={mean_train_loss:.4f} "
            f"val_macro_f1={val_macro:.4f} "
            f"val_acc={val_acc:.2f}%"
        )

        if (
            val_macro > best_val_macro_f1
            or (
                abs(
                    val_macro
                    - best_val_macro_f1
                ) < 1e-8
                and val_acc > best_val_acc
            )
        ):
            best_val_macro_f1 = val_macro
            best_val_acc = val_acc
            best_epoch = int(epoch)
            best_state = {
                k: v.detach().cpu().clone()
                for k, v
                in model.state_dict().items()
            }

        # Save after every epoch so a kernel interruption does not lose the run.
        torch.save(
            {
                "config": cfg,
                "completed_epoch": int(epoch),
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_epoch": best_epoch,
                "best_val_macro_f1": best_val_macro_f1,
                "best_val_acc": best_val_acc,
                "best_state_dict": best_state,
                "history": history,
                "fixed_budget_epochs": int(NUM_EPOCHS),
                "full_source_training_split": True,
            },
            resume_path,
        )

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if best_state is None:
        raise RuntimeError(
            f"No valid checkpoint selected for {config_id}"
        )

    ckpt_path = os.path.join(
        out_dir,
        f"{config_id}_best.pt",
    )

    final_epoch_row = history[-1]

    torch.save(
        {
            "config": cfg,
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_macro_f1,
            "best_val_acc": best_val_acc,
            "final_epoch": int(
                final_epoch_row["epoch"]
            ),
            "final_epoch_val_macro_f1": float(
                final_epoch_row["val_macro_f1"]
            ),
            "final_epoch_val_acc": float(
                final_epoch_row["val_accuracy_pct"]
            ),
            "model_state_dict": best_state,
            "source_validation_only": True,
            "fixed_budget_epochs": int(NUM_EPOCHS),
            "full_source_training_split": True,
        },
        ckpt_path,
    )

    # Finished run: the permanent best checkpoint + history are sufficient.
    if os.path.exists(resume_path):
        os.remove(resume_path)

    del model, optimizer, criterion, weights
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        **cfg,
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_macro_f1,
        "best_val_accuracy_pct": best_val_acc,
        "final_epoch_val_macro_f1": float(
            final_epoch_row["val_macro_f1"]
        ),
        "final_epoch_val_accuracy_pct": float(
            final_epoch_row["val_accuracy_pct"]
        ),
        "checkpoint": ckpt_path,
        "history_csv": history_path,
    }



def evaluate_reported_baseline(dataset, train_idx, val_idx):
    if not os.path.exists(REPORTED_MOTIF_CKPT):
        raise FileNotFoundError(
            f"Reported motif checkpoint not found: {REPORTED_MOTIF_CKPT}\n"
            "Update REPORTED_MOTIF_RESULTS_DIR only if your existing retained checkpoint lives elsewhere."
        )
    model = RobustMosaicGNN(
        hidden_dim=HIDDEN_DIM,
        num_phases=NUM_PHASES,
        motif_dim=MOTIF_DIM,
    ).to(DEVICE)
    ckpt = torch.load(REPORTED_MOTIF_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model.motif_window = 16
    model.motif_stride = 8
    weights = compute_motif_class_weights(dataset, train_idx, 16, 8, NUM_PHASES).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTHING)
    ev = evaluate_split(model, dataset, val_idx, DEVICE, criterion, PHASE_NAMES, AUX_WEIGHT, SEQ_WEIGHT)
    return {
        "config_id": "retained_baseline",
        "comparison_group": "baseline",
        "initialization": "phase_checkpoint",
        "freeze_first_gatv2": True,
        "motif_window": 16,
        "motif_stride": 8,
        "best_epoch": int(ckpt.get("epoch", -1)),
        "best_val_macro_f1": float(ev["macro_f1"]),
        "best_val_accuracy_pct": float(ev["accuracy"] * 100.0),
        "checkpoint": REPORTED_MOTIF_CKPT,
    }



def build_deterministic_screen_subset(train_idx):
    """
    Return the complete saved source-training split in one deterministic order.
    The function name is retained only to minimize changes to the audited code path.
    """
    train_idx = np.asarray(train_idx, dtype=int)
    return np.sort(train_idx)


def load_existing_screen_rows(progress_csv):
    if not os.path.exists(progress_csv):
        return []

    df = pd.read_csv(progress_csv)
    required = {
        "config_id",
        "best_val_macro_f1",
        "best_val_accuracy_pct",
    }

    if not required.issubset(df.columns):
        return []

    return df.to_dict("records")


def build_screen_comparison_table(df):
    lookup = df.set_index("config_id")
    baseline_id = "baseline_L16_S8_phaseinit_frozen"

    comparisons = [
        (
            "first GATv2 freezing",
            baseline_id,
            "phase_init_full_unfreeze",
        ),
        (
            "phase-checkpoint initialization",
            "phase_init_full_unfreeze",
            "random_init_full_unfreeze",
        ),
        (
            "window length L=8 vs retained L=16",
            baseline_id,
            "window_8_stride_8",
        ),
        (
            "window length L=24 vs retained L=16",
            baseline_id,
            "window_24_stride_8",
        ),
        (
            "stride 4 vs retained stride 8",
            baseline_id,
            "window_16_stride_4",
        ),
        (
            "stride 16 vs retained stride 8",
            baseline_id,
            "window_16_stride_16",
        ),
    ]

    rows = []

    for factor, reference, comparison in comparisons:
        if (
            reference not in lookup.index
            or comparison not in lookup.index
        ):
            continue

        ref_macro = float(
            lookup.loc[
                reference,
                "best_val_macro_f1",
            ]
        )
        cmp_macro = float(
            lookup.loc[
                comparison,
                "best_val_macro_f1",
            ]
        )
        ref_acc = float(
            lookup.loc[
                reference,
                "best_val_accuracy_pct",
            ]
        )
        cmp_acc = float(
            lookup.loc[
                comparison,
                "best_val_accuracy_pct",
            ]
        )

        rows.append(
            {
                "sensitivity_factor": factor,
                "reference": reference,
                "comparison": comparison,
                "reference_val_macro_f1": ref_macro,
                "comparison_val_macro_f1": cmp_macro,
                "comparison_minus_reference_macro_f1_pp": (
                    100.0 * (cmp_macro - ref_macro)
                ),
                "reference_val_accuracy_pct": ref_acc,
                "comparison_val_accuracy_pct": cmp_acc,
                "comparison_minus_reference_accuracy_pp": (
                    cmp_acc - ref_acc
                ),
            }
        )

    return pd.DataFrame(rows)


def run_motif_extractor_fixed_budget_sensitivity():
    set_seed(SCREEN_SEED)
    os.makedirs(
        AUDIT_RESULTS_DIR,
        exist_ok=True,
    )

    print(f"[DEVICE] {DEVICE}")
    print(f"[AUDIT DIR] {AUDIT_RESULTS_DIR}")
    print(
        "[DESIGN] 5 epochs/configuration, full saved source-training split, "
        "full source-validation split."
    )
    print(
        "[BOUNDARY] Source-test and remote outcomes are not used."
    )
    print(
        "[PROTOCOL] 7 unique configurations; no factorial grid; "
        "best validation Macro-F1 within the common 5-epoch budget is primary."
    )

    required_paths = {
        "BASE_PATH": BASE_PATH,
        "BIN_PATH": BIN_PATH,
        "PHASE_CKPT": PHASE_CKPT,
        "SPLIT_NPZ": SPLIT_NPZ,
    }

    missing = []

    print(
        "\n[PREFLIGHT — SOURCE DEVELOPMENT ARTIFACTS]"
    )

    for name, path in required_paths.items():
        ok = os.path.exists(path)
        print(
            f"{name:22s}: "
            f"{'OK' if ok else 'MISSING'}  {path}"
        )

        if not ok:
            missing.append(
                (name, path)
            )

    if missing:
        message = "\n".join(
            f"  - {name}: {path}"
            for name, path in missing
        )
        raise FileNotFoundError(
            "Required source artifact(s) are missing:\n"
            + message
            + "\nNo sensitivity screen has started."
        )

    dataset = SurgicalTurboDataset(
        BASE_PATH,
        BIN_PATH,
        chunk_size=CHUNK_SIZE,
        force_reprocess=False,
    )

    train_idx, val_idx, test_idx = load_saved_split(
        SPLIT_NPZ,
        len(dataset),
    )

    screen_train_idx = build_deterministic_screen_subset(
        train_idx
    )

    print(
        f"[SPLIT] original_train={len(train_idx)} "
        f"audit_train={len(screen_train_idx)} "
        f"val={len(val_idx)} "
        f"test={len(test_idx)} (test not used)"
    )

    progress_csv = os.path.join(
        AUDIT_RESULTS_DIR,
        "motif_extractor_fixed_budget_results.csv",
    )
    comparison_csv = os.path.join(
        AUDIT_RESULTS_DIR,
        "motif_extractor_fixed_budget_comparisons.csv",
    )

    existing_rows = load_existing_screen_rows(
        progress_csv
    )
    completed_ids = {
        str(row["config_id"])
        for row in existing_rows
    }

    if completed_ids:
        print(
            "\n[RESUME] already completed:",
            sorted(completed_ids),
        )

    rows = list(existing_rows)

    for cfg in AUDIT_CONFIGS:
        config_id = cfg["config_id"]

        if config_id in completed_ids:
            print(
                f"[SKIP COMPLETED] {config_id}"
            )
            continue

        # Reset seed before every configuration so ordering/randomness is controlled.
        set_seed(SCREEN_SEED)

        result = train_one_audit_variant(
            cfg,
            dataset,
            screen_train_idx,
            val_idx,
            AUDIT_RESULTS_DIR,
        )

        rows.append(result)

        # Save immediately after every finished configuration.
        progress_df = pd.DataFrame(rows)

        # Stable order matching AUDIT_CONFIGS.
        order = {
            cfg["config_id"]: i
            for i, cfg in enumerate(AUDIT_CONFIGS)
        }
        progress_df["_order"] = (
            progress_df["config_id"]
            .map(order)
        )
        progress_df = (
            progress_df
            .sort_values("_order")
            .drop(columns=["_order"])
            .reset_index(drop=True)
        )

        progress_df.to_csv(
            progress_csv,
            index=False,
        )

        comparison_df = build_screen_comparison_table(
            progress_df
        )
        comparison_df.to_csv(
            comparison_csv,
            index=False,
        )

        print(
            f"\n[SAVED AFTER {config_id}] "
            f"{progress_csv}"
        )

    final_df = pd.read_csv(
        progress_csv
    )

    final_comparisons = build_screen_comparison_table(
        final_df
    )
    final_comparisons.to_csv(
        comparison_csv,
        index=False,
    )

    protocol = {
        "purpose": (
            "Local source-validation "
            "optimization sensitivity audit"
        ),
        "scope": "fixed-budget one-factor-at-a-time source-validation ablation",
        "seed": int(SCREEN_SEED),
        "epochs_per_configuration": int(NUM_EPOCHS),
        "source_training_fraction": 1.0,
        "same_full_source_training_split_for_all_configurations": True,
        "validation_split": "full saved source-validation split",
        "source_test_used": False,
        "remote_used": False,
        "source_test_or_remote_used_for_selection": False,
    }

    protocol_path = os.path.join(
        AUDIT_RESULTS_DIR,
        "motif_extractor_fixed_budget_protocol.json",
    )

    with open(
        protocol_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            protocol,
            f,
            indent=2,
        )

    print(
        "\n[FINAL FIXED-BUDGET SCREEN]"
    )
    print(
        final_df[
            [
                "config_id",
                "best_epoch",
                "best_val_macro_f1",
                "best_val_accuracy_pct",
                "final_epoch_val_macro_f1",
                "final_epoch_val_accuracy_pct",
            ]
        ].to_string(index=False)
    )

    print(
        "\n[FACTOR-MATCHED COMPARISONS]"
    )
    print(
        final_comparisons.to_string(
            index=False
        )
    )

    print(f"\nSaved: {progress_csv}")
    print(f"Saved: {comparison_csv}")
    print(f"Saved: {protocol_path}")

    return final_df


if __name__ == "__main__":
    run_motif_extractor_fixed_budget_sensitivity()



## How the results should be read

The publication comparison is the factor-matched table saved as:

`motif_extractor_fixed_budget_comparisons.csv`

The raw seven-configuration summary is:

`motif_extractor_fixed_budget_results.csv`

Interpret each comparison only within the common five-epoch source-validation budget. The experiment is designed to answer the optimization-sensitivity question; it is not a factorial search over combinations.

For window length and stride, the class-weighting **rule** is held fixed, while the actual inverse-frequency weights are recomputed from the source-training motif windows produced by each temporal setting. This preserves the original training procedure under each tested window configuration.

## Manuscript-facing expected result

The run reproduces **Table 5** of the manuscript. The manuscript table reports the best source-validation macro F1 observed within the common five-epoch budget for each matched configuration.

A compact reference is included in `expected_results/table5_motif_extractor_sensitivity.csv`.
